In [ ]:
import os

os.environ["SOCCERDATA_DIR"]="../soccerdata_local"

import soccerdata as sd

scraper = sd.Sofascore(leagues=None)

metodos_lectura = [metodo for metodo in dir(scraper) if metodo.startswith('read_')]

print("Métodos disponibles Sofascore:")
for m in metodos_lectura:
    print(f"- {m}")


In [ ]:
import requests

#API Sofascore
url = "https://api.sofascore.com/api/v1/config/unique-tournaments/EN/football"

cabeceras = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "*/*",
    "Origin": "https://www.sofascore.com",
    "Referer": "https://www.sofascore.com/"
}


respuesta = requests.get(url, headers=cabeceras)

print(f"Código de estado: {respuesta.status_code}")

Código de estado: 403


In [ ]:
from curl_cffi import requests

url = "https://api.sofascore.com/api/v1/config/unique-tournaments/EN/football"


respuesta = requests.get(url, impersonate="chrome")

print(f"Código de estado: {respuesta.status_code}")
if respuesta.status_code == 200:
    print("¡Perro guardián superado! Estamos dentro de la API.")

Código de estado: 404


In [ ]:
from curl_cffi import requests
import pandas as pd
import os

url_jugador = "https://www.sofascore.com/api/v1/event/16384052/player/1101427/rating-breakdown"

respuesta = requests.get(url_jugador, impersonate="chrome")

print(f"Código de estado: {respuesta.status_code}")

if respuesta.status_code == 200:
    datos = respuesta.json()
    print("Datos del jugador:")
    print(str(datos)[:500])
else:
    print("Algo falló:", respuesta.text)


lista_pases = datos['passes']

# 'json_normalize' aplana los diccionarios.'playerCoordinates': {'x': 98.4} -> Columna 'playerCoordinates.x'
df_pases = pd.json_normalize(lista_pases)

os.makedirs("../data/bronze", exist_ok=True)
ruta_archivo = "../data/bronze/hasselt_pases_raw.parquet"
df_pases.to_parquet(ruta_archivo)

print(f"Datos guardados en: {ruta_archivo}")
print(df_pases.head(3))

Código de estado: 200
¡Bingo! Hemos robado los datos del jugador:
{'passes': [{'playerCoordinates': {'x': 98.4, 'y': 74}, 'passEndCoordinates': {'x': 96.7, 'y': 56.6}, 'eventActionType': 'cross', 'isHome': True, 'keypass': False}, {'playerCoordinates': {'x': 75.3, 'y': 70.9}, 'passEndCoordinates': {'x': 75.5, 'y': 63.4}, 'eventActionType': 'pass', 'isHome': True, 'outcome': True, 'keypass': False}, {'playerCoordinates': {'x': 92.8, 'y': 73.2}, 'passEndCoordinates': {'x': 93.2, 'y': 70.3}, 'eventActionType': 'cross', 'isHome': True, 'keypass': False}, {'playerC
¡Éxito! Datos guardados físicamente en: ../data/bronze/hasselt_pases_raw.parquet

Así luce tu Capa Bronce (Primeras 3 filas):
  eventActionType  isHome  keypass  playerCoordinates.x  playerCoordinates.y  \
0           cross    True    False                 98.4                 74.0   
1            pass    True    False                 75.3                 70.9   
2           cross    True    False                 92.8            

In [ ]:
import time
import random
import pandas as pd
from curl_cffi import requests
import os

liga_id = 9
temporada_id = 96912
ronda = 1
partidos_totales = []

print("Empezando extracción del calendario...")

# Bucle infinito que romperemos cuando se acaben las jornadas
while True:
    url = f"https://www.sofascore.com/api/v1/unique-tournament/{liga_id}/season/{temporada_id}/events/round/{ronda}"
    
    respuesta = requests.get(url, impersonate="chrome")
    
    if respuesta.status_code != 200:
        print(f"Ronda {ronda} no encontrada. Fin del calendario.")
        break
        
    datos = respuesta.json()
    
    if 'events' not in datos or len(datos['events']) == 0:
        print(f"Jornada {ronda} vacía. Fin del calendario.")
        break
        
    eventos = datos['events']
    print(f"Descargando Ronda {ronda}: {len(eventos)} partidos encontrados...")
    
    # Extraemos solo lo que nos importa
    for partido in eventos:
        partidos_totales.append({
            "match_id": partido.get("id"),
            "home_team": partido.get("homeTeam", {}).get("name"),
            "away_team": partido.get("awayTeam", {}).get("name"),
            "round": ronda,
            # Guardamos el estado
            "status": partido.get("status", {}).get("description") 
        })
        
    ronda += 1
    
    time.sleep(random.uniform(1.0, 2.5))

df_calendario = pd.DataFrame(partidos_totales)

os.makedirs("../data/bronze", exist_ok=True)
ruta_calendario = "../data/bronze/calendario_challenger.parquet"

df_calendario.to_parquet(ruta_calendario)

print(f"\nExtracción terminada.")
print(f"Total de partidos guardados: {len(df_calendario)}")
print(f"Archivo guardado en: {ruta_calendario}")

Empezando extracción del calendario...
Descargando Ronda 1: 7 partidos encontrados...
Descargando Ronda 2: 7 partidos encontrados...
Descargando Ronda 3: 7 partidos encontrados...
Descargando Ronda 4: 7 partidos encontrados...
Descargando Ronda 5: 7 partidos encontrados...
Descargando Ronda 6: 7 partidos encontrados...
Descargando Ronda 7: 7 partidos encontrados...
Descargando Ronda 8: 7 partidos encontrados...
Descargando Ronda 9: 7 partidos encontrados...
Descargando Ronda 10: 7 partidos encontrados...
Descargando Ronda 11: 7 partidos encontrados...
Descargando Ronda 12: 7 partidos encontrados...
Descargando Ronda 13: 7 partidos encontrados...
Descargando Ronda 14: 7 partidos encontrados...
Descargando Ronda 15: 7 partidos encontrados...
Descargando Ronda 16: 7 partidos encontrados...
Descargando Ronda 17: 7 partidos encontrados...
Descargando Ronda 18: 7 partidos encontrados...
Descargando Ronda 19: 7 partidos encontrados...
Descargando Ronda 20: 7 partidos encontrados...
Descargand

In [5]:


import time
import random
import pandas as pd
from curl_cffi import requests
import os

jornada = 1

calendario = pd.read_parquet("../data/bronze/calendario_challenger.parquet")

jornada_actual = calendario[calendario['round'] == jornada].get('match_id').tolist()

lineups = []

match_id = jornada_actual[0] if jornada_actual else None

if match_id:
    url = f"https://www.sofascore.com/api/v1/event/16384051/lineups"
    
    respuesta = requests.get(url, impersonate="chrome")
    
    if respuesta.status_code == 200:
        datos_lineups = respuesta.json()
        print(f"Lineups para el partido {match_id}:")
        print(datos_lineups)
    else:
        print(f"No se pudo obtener lineups para el partido {match_id}. Código de estado: {respuesta.status_code}")

lineups.append(datos_lineups)





Lineups para el partido 16384051:
{'confirmed': True, 'home': {'players': [{'player': {'name': 'Leonardo Cascio', 'firstName': 'Leonardo Cascio', 'lastName': '', 'slug': 'leonardo-cascio', 'shortName': 'L. Cascio', 'position': 'G', 'jerseyNumber': '99', 'height': 188, 'userCount': 18, 'gender': 'M', 'sofascoreId': 'hir4t9', 'country': {'alpha2': 'BE', 'alpha3': 'BEL', 'name': 'Belgium', 'slug': 'belgium'}, 'id': 1155612, 'marketValueCurrency': 'EUR', 'dateOfBirthTimestamp': 1088294400, 'fieldTranslations': {'nameTranslation': {'ar': 'أمارتي، هارشكا', 'ru': 'Леонардо Кассио'}, 'shortNameTranslation': {'ar': 'ل. كاسكيو', 'ru': 'Л. Кассио'}}}, 'teamId': 390768, 'shirtNumber': 99, 'jerseyNumber': '99', 'position': 'G', 'substitute': False, 'statistics': {'totalPass': 33, 'accuratePass': 21, 'totalLongBalls': 18, 'accurateLongBalls': 6, 'goalAssist': 0, 'accurateOwnHalfPasses': 20, 'totalOwnHalfPasses': 21, 'accurateOppositionHalfPasses': 1, 'totalOppositionHalfPasses': 12, 'aerialWon': 1, 

In [ ]:
import time
import random
import pandas as pd
from curl_cffi import requests
import os

jornada = 1

calendario = pd.read_parquet("../data/bronze/calendario_challenger.parquet")
lineups_limpios = []
while True:
    jornada_actual = calendario[
        (calendario['round'] == jornada) & 
        (calendario['status'] == 'Ended')
    ]['match_id'].tolist()


    if len(jornada_actual) > 0:
        print(f"Partidos encontrados para la jornada {jornada}: {len(jornada_actual)}")
        
    else:
        print(f"No hay partidos finalizados para la jornada {jornada}.")
        break
    for match_id in jornada_actual:
        url = f"https://www.sofascore.com/api/v1/event/{match_id}/lineups"
        
        respuesta = requests.get(url, impersonate="chrome")
        
        if respuesta.status_code == 200:
            datos_lineups = respuesta.json()
            
            #Normalizamos los datos JSON
            for equipo in ['home', 'away']:
                lista_jugadores = datos_lineups.get(equipo, {}).get('players', [])
                
                for j in lista_jugadores:
                    lineups_limpios.append({
                        "round": jornada,
                        "match_id": match_id,
                        "team_side": equipo,
                        "player_id": j['player']['id'],
                        "player_name": j['player']['name'],
                        "position": j.get('position', 'N/A'),
                        "is_starter": not j.get('substitute', False),
                        "minutes_played": j.get('statistics', {}).get('minutesPlayed', 0)
                    })
            print(f"✅ Alineaciones extraídas para el partido {match_id}")
        else:
            print(f"❌ Error al obtener lineups del partido {match_id}. Estado: {respuesta.status_code}")
            
        time.sleep(random.uniform(1.0, 2.5))
    jornada += 1

df_lineups = pd.DataFrame(lineups_limpios)
ruta_lineups = f"../data/bronze/lineups.parquet"
df_lineups.to_parquet(ruta_lineups)

print(f"\nScript finalizado. Guardado en {ruta_lineups}")
print(f"Total de filas (jugadores): {len(df_lineups)}")
print("\nPrimeras 5 filas:")
print(df_lineups.head())

In [ ]:
import pandas as pd
import os
import time
import random
from curl_cffi import requests

lineups = pd.read_parquet("../data/bronze/lineups.parquet")
lineups_activos = lineups[lineups['minutes_played'] > 0]

carpeta_eventos = "../data/bronze/eventos_raw"
os.makedirs(carpeta_eventos, exist_ok=True)

print(f"Total de jugadores a extraer: {len(lineups_activos)}")

for index, row in lineups_activos.iterrows():
    match_id = row['match_id']
    player_id = row['player_id']
    
    ruta_archivo = f"{carpeta_eventos}/match_{match_id}_player_{player_id}.parquet"
    
    if os.path.exists(ruta_archivo):
        continue
        
    url = f"https://www.sofascore.com/api/v1/event/{match_id}/player/{player_id}/rating-breakdown"
    
    try:
        respuesta = requests.get(url, impersonate="chrome")
        
        if respuesta.status_code == 200:
            datos = respuesta.json()
            
            dataframes_temporales = []
            
            for tipo_evento, contenido in datos.items():
                
                if isinstance(contenido, list) and len(contenido) > 0:
                    
                    df_temp = pd.json_normalize(contenido)
                    df_temp['event_category'] = tipo_evento 
                    dataframes_temporales.append(df_temp)
            
            if dataframes_temporales:
                df_eventos_jugador = pd.concat(dataframes_temporales, ignore_index=True)
                
                df_eventos_jugador['match_id'] = match_id
                df_eventos_jugador['player_id'] = player_id
                
                df_eventos_jugador.to_parquet(ruta_archivo)
                print(f"✅ Partido {match_id} | Jugador {player_id}: {len(df_eventos_jugador)} eventos totales guardados.")
            else:
                print(f"⚠️ Sin eventos registrados: Partido {match_id} | Jugador {player_id}")
                
        else:
            print(f"Error {respuesta.status_code} en Partido {match_id} | Jugador {player_id}")
            
    except Exception as e:
        print(f"❌ Fallo de conexión en Partido {match_id} | Jugador {player_id}: {e}")
        
    time.sleep(random.uniform(1.0, 2.0))

print("\nExtracción completada.")